In [ ]:

import pandas as pd
from IPython.display import display

ventas = pd.read_csv("ventas.csv")
productos = pd.read_csv("productos_.csv")
clientes = pd.read_csv("clientes.csv")

ventas["fecha"] = pd.to_datetime(ventas["fecha"])
ventas["importe"] = ventas["cantidad"] * ventas["precio_unitario"]

def dolares(valor):
    return f"${valor:,.2f}"

# Mostrar los datos
print("VENTAS")
display(ventas.head())

print("PRODUCTOS")
display(productos.head())

print("CLIENTES")
display(clientes.head())


VENTAS


,id_linea,id_ticket,fecha,sucursal,id_cliente,id_producto,cantidad,precio_unitario,metodo_pago,importe
0,L00001,V0001,2025-11-01,Sur,C089,P087,1,1.50,Transferencia,1.50
1,L00002,V0001,2025-11-01,Sur,C089,P326,4,0.93,Transferencia,3.72
2,L00003,V0001,2025-11-01,Sur,C089,P282,1,1.12,Transferencia,1.12
3,L00004,V0001,2025-11-01,Sur,C089,P088,2,1.64,Transferencia,3.28
4,L00005,V0001,2025-11-01,Sur,C089,P14,1,2.20,Transferencia,2.20


PRODUCTOS


,id_producto,producto,categoria,precio_lista
0,P07,Resaltador,Escritura,1.1
1,P12,Paquete de fundas A4,Archivo,2.1
2,P13,Regla de 30 cm,Accesorios,0.8
3,P05,Esferográfico azul,Escritura,0.6
4,P06,Lápiz HB,Escritura,0.4


CLIENTES


,id,nombre,ciudad,tipo_cliente
0,C001,Sofía Villacís,Quito,Docente
1,C002,Daniel Vega,Quito,Docente
2,C003,Nicolás Sánchez,Sangolquí,Estudiante
3,C004,Daniel Romero,Quito,Estudiante
4,C005,Sebastián Yánez,Cayambe,Estudiante


**Preguta 1.**
Líneas, tickets y unidades. ¿Cuántas líneas de venta, cuántos tickets distintos y cuántas unidades vendidas hay en el archivo? Explique qué representa cada conteo.

In [4]:
lineas = len(ventas)
tickets = ventas["id_ticket"].nunique()
unidades = ventas["cantidad"].sum()

display(pd.DataFrame({
    "Indicador": ["Líneas de venta", "Tickets distintos", "Unidades vendidas"],
    "Total": [lineas, tickets, unidades]
}))
print("Una línea es un producto registrado en un ticket; un ticket es una compra y una unidad es un artículo vendido.")

,Indicador,Total
0,Líneas de venta,2100
1,Tickets distintos,300
2,Unidades vendidas,3880


Una línea es un producto registrado en un ticket; un ticket es una compra y una unidad es un artículo vendido.


**Pregunta 2.**
Tickets por sucursal. ¿Cuántos tickets se registraron en cada sucursal? ¿Cuál registró más?

In [6]:
p2 = (ventas.groupby("sucursal")["id_ticket"]
      .nunique().sort_values(ascending=False))
display(p2.rename("Tickets").reset_index())
print(f"Sucursal con más tickets: {p2.index[0]} ({p2.iloc[0]} tickets).")

,sucursal,Tickets
0,Central,146
1,Norte,83
2,Sur,71


Sucursal con más tickets: Central (146 tickets).


**Pregunta 3.**
Métodos de pago. ¿Cuántos tickets se pagaron con cada método y cuántos tienen ese dato sin registrar?

In [8]:
pagos = ventas[["id_ticket", "metodo_pago"]].drop_duplicates("id_ticket").copy()
pagos["metodo_pago"] = pagos["metodo_pago"].fillna("Sin registrar")
p3 = pagos.groupby("metodo_pago").size().sort_values(ascending=False)
display(p3.rename("Tickets").reset_index())
print(f"Tickets sin método registrado: {p3.get('Sin registrar', 0)}.")

,metodo_pago,Tickets
0,Efectivo,144
1,Tarjeta,84
2,Transferencia,48
3,Sin registrar,24


Tickets sin método registrado: 24.


In [ ]:
**Pregunta 4**
Precios de lista por categoría. ¿Cuál es el precio de lista mínimo, máximo y promedio de los productos de cada categoría? Use el catálogo de productos.

In [20]:
p4 = (productos.groupby('categoria')['precio_lista']
           .agg(precio_promedio='mean', precio_minimo='min', precio_maxima='max')
           .round(2).reset_index())
display(p4)

,categoria,precio_promedio,precio_minimo,precio_maxima
0,Accesorios,1.62,0.40,22.90
1,Archivo,1.67,0.35,3.95
2,Escritura,1.35,0.40,3.75
3,Papel y cuadernos,1.67,0.28,5.75


**Pregunta 5**
Unidades por sucursal y categoría. ¿Cuántas unidades vendió cada sucursal de cada categoría? Presente una tabla cruzada con totales por fila y por columna. Compruebe que el total general coincide con la suma de cantidad en ventas.csv.



In [17]:
ventas_categoria = ventas.merge(
    productos[["id_producto", "categoria"]],
    on="id_producto", how="left", validate="many_to_one"
)
p5 = pd.pivot_table(
    ventas_categoria, index="sucursal", columns="categoria",
    values="cantidad", aggfunc="sum", fill_value=0,
    margins=True, margins_name="Total"
)
display(p5)
assert p5.loc["Total", "Total"] == unidades
print(f"Comprobación: {p5.loc['Total', 'Total']} unidades = suma de cantidad ({unidades}).")

categoria,Accesorios,Archivo,Escritura,Papel y cuadernos,Total
sucursal,,,,,
Central,453,481,437,524,1895
Norte,256,264,306,259,1085
Sur,191,223,238,248,900
Total,900,968,981,1031,3880


Comprobación: 3880 unidades = suma de cantidad (3880).


**Pregunta 6.**
Importe de ventas por sucursal. ¿Cuál fue el importe total de ventas de cada sucursal? Calcule primero el precio total de cada línea como cantidad × precio_unitario.

In [19]:
p6 = (ventas.groupby("sucursal")["importe"]
      .sum().sort_values(ascending=False))
display(p6.map(dolares).rename("Importe total (USD)").reset_index())
print(f"Importe general de ventas: {dolares(p6.sum())}.")

,sucursal,Importe total (USD)
0,Central,"$3,025.04"
1,Norte,"$1,635.35"
2,Sur,"$1,443.24"


Importe general de ventas: $6,103.63.


**Pregunta 7.**
Tamaño de las compras. ¿Cuántos tickets corresponden a compras de hasta 7 unidades, de 8 a 14 unidades y de 15 o más unidades? Sume primero las cantidades de todas las líneas de cada ticket.

In [21]:
unidades_ticket = ventas.groupby("id_ticket")["cantidad"].sum()
rangos = pd.cut(
    unidades_ticket, bins=[float("-inf"), 7, 14, float("inf")],
    labels=["Hasta 7 unidades", "De 8 a 14 unidades", "15 o más unidades"]
)
p7 = rangos.value_counts(sort=False)
display(p7.rename("Tickets").reset_index().rename(columns={"cantidad": "Tamaño de compra"}))
assert p7.sum() == tickets
print(f"Total clasificado: {p7.sum()} tickets.")


,Tamaño de compra,Tickets
0,Hasta 7 unidades,39
1,De 8 a 14 unidades,153
2,15 o más unidades,108


Total clasificado: 300 tickets.


**Pregunta 8.**
Ventas por mes. ¿Cuántos tickets y qué importe total se registraron en cada mes? ¿En qué mes se alcanzó el mayor importe?

In [22]:
ventas["mes"] = ventas["fecha"].dt.strftime("%Y-%m")
p8 = (ventas.groupby("mes")
      .agg(Tickets=("id_ticket", "nunique"),
           Importe=("importe", "sum"))
      .sort_index())
reporte8 = p8.copy()
reporte8["Importe"] = reporte8["Importe"].map(dolares)
display(reporte8.rename(columns={"Importe": "Importe total (USD)"}).reset_index())
mes_max = p8["Importe"].idxmax()
print(f"Mes con mayor importe: {mes_max}, con {dolares(p8.loc[mes_max, 'Importe'])}.")

,mes,Tickets,Importe total (USD)
0,2025-11,25,$492.51
1,2025-12,25,$645.54
2,2026-01,25,$447.48
3,2026-02,25,$466.36
4,2026-03,25,$595.33
5,2026-04,25,$505.43
6,2026-05,25,$453.94
7,2026-06,25,$500.54
8,2026-07,25,$499.88
9,2026-08,25,$478.79


Mes con mayor importe: 2025-12, con $645.54.


**Preguta 9.**
Producto más vendido por sucursal. ¿Qué producto acumuló más unidades vendidas en cada sucursal? Indique su código, nombre y cantidad vendida. Si hay un empate, incluya todos los productos empatados.

In [24]:
unidades_producto = (ventas.groupby(["sucursal", "id_producto"], as_index=False)
                    .agg(unidades_vendidas=("cantidad", "sum")))
ganadores = unidades_producto[
    unidades_producto["unidades_vendidas"].eq(
        unidades_producto.groupby("sucursal")["unidades_vendidas"].transform("max")
    )
]
p9 = (ganadores.merge(productos[["id_producto", "producto"]],
                     on="id_producto", validate="many_to_one")
      [["sucursal", "id_producto", "producto", "unidades_vendidas"]]
      .sort_values(["sucursal", "id_producto"]))
display(p9.reset_index(drop=True))

,sucursal,id_producto,producto,unidades_vendidas
0,Central,P399,Estuche escolar (gris),17
1,Norte,P141,Marcador de punta fina (naranja),17
2,Sur,P016,Cuaderno cuadriculado A5 (24 hojas),11
3,Sur,P304,Portadocumentos con cierre (rojo),11


**Pregunta 10.**
Compras por tipo de cliente. Para cada tipo de cliente, ¿cuántos clientes distintos compraron, cuántos tickets generaron y cuál fue el importe total de sus compras?

In [23]:
ventas_clientes = ventas.merge(
    clientes[["id", "tipo_cliente"]],
    left_on="id_cliente", right_on="id",
    how="left", validate="many_to_one"
)
p10 = (ventas_clientes.groupby("tipo_cliente")
       .agg(Clientes_distintos=("id_cliente", "nunique"),
            Tickets=("id_ticket", "nunique"),
            Importe=("importe", "sum"))
       .sort_index())
reporte10 = p10.copy()
reporte10["Importe"] = reporte10["Importe"].map(dolares)
display(reporte10.rename(columns={
    "Clientes_distintos": "Clientes que compraron",
    "Importe": "Importe total (USD)"
}).reset_index())

,tipo_cliente,Clientes que compraron,Tickets,Importe total (USD)
0,Docente,16,46,$899.74
1,Estudiante,72,226,"$4,688.00"
2,Personal administrativo,12,28,$515.89
